# Tool Use, Function Calling & MCP

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/09-tool-use-and-mcp

We build a tiny ReAct tool-using loop, then quantify why agents compound errors and how retries fight back.

Self-contained: NumPy + matplotlib only. No torch, no sklearn, no network, no API keys.

> **To save your work:** click **Copy to Drive** at the top, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. A toy tool-calling loop

The model never runs tools — it *emits a call* that our runtime executes. We mock a model that parses a question into tool calls (`search`, `calc`) and feed observations back. The point is the control flow, not a real LLM.

In [ ]:
TOOLS = {
    'capital': {'France': 'Paris', 'Japan': 'Tokyo'},
    'population': {'Paris': 2_100_000, 'Tokyo': 14_000_000},
}

def run_tool(name, arg):
    """Your runtime executes the tool the 'model' requested."""
    if name == 'calc':
        return eval(arg, {'__builtins__': {}})
    return TOOLS[name].get(arg, 'UNKNOWN')

# A scripted 'agent': a list of (thought, tool, arg) emitted step by step.
trajectory = [
    ('need the capital of France', 'capital', 'France'),
    ('now its population',         'population', 'Paris'),
    ('divide by 1000',            'calc', '2100000 / 1000'),
]
obs = None
for thought, tool, arg in trajectory:
    print(f'Thought: {thought:32s} Action: {tool}({arg!r})')
    obs = run_tool(tool, arg)
    print(f'   Observation: {obs}')
print('Answer:', obs)

## 2. Why reliability is $p^n$

If each Thought→Action→Observation step succeeds with probability $p$, an $n$-step trajectory succeeds with probability $p^n$. We plot it: a 95%-reliable step decays fast.

In [ ]:
n = np.arange(0, 21)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for p in [0.99, 0.95, 0.90, 0.80]:
    ax.plot(n, p**n * 100, 'o-', lw=2, markersize=4, label=f'p = {p:.2f}')
ax.axhline(50, color=ROSE, ls='--', lw=1, alpha=0.6, label='50%')
ax.set_xlabel('trajectory length n (steps)')
ax.set_ylabel('end-to-end success (%)')
ax.set_title('Agents compound errors: success = p^n')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('At p=0.95, 10 steps:', round(0.95**10 * 100, 1), '%')

## 3. Retries recover a flaky step

If a tool fails transiently with probability $f$ per attempt, allowing $k$ attempts makes that step succeed with probability $1-f^{k}$. Three retries turn a 60%-failure step into a reliable one.

In [ ]:
f = 0.6  # per-attempt failure of one flaky tool
ks = np.arange(1, 7)
step_success = 1 - f**ks
for k, s in zip(ks, step_success):
    print(f'{k} attempts: step success = {s:.1%}')

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(ks, step_success * 100, 'o-', color=TEAL, lw=2, markersize=8)
ax.set_xlabel('attempts allowed (k)'); ax.set_ylabel('step success (%)')
ax.set_title('Retries turn a 60%-failure step reliable'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## ✏️ Your turn — compounded reliability

Implement `trajectory_reliability(p, n)` = the probability an $n$-step agent succeeds end-to-end when each step is `p` reliable.

In [ ]:
def trajectory_reliability(p, n):
    """TODO(you): return p**n."""
    # TODO
    return ...


In [ ]:
r10 = trajectory_reliability(0.95, 10)
r1 = trajectory_reliability(0.99, 1)
print('p=0.95, n=10 ->', round(r10, 4), '(expected ~0.5987)')
assert abs(r10 - 0.95**10) < 1e-9
assert abs(r1 - 0.99) < 1e-9
assert abs(trajectory_reliability(1.0, 100) - 1.0) < 1e-9
print('\n✅ reliability compounds as expected.')

<details>
<summary>Solution</summary>

```python
def trajectory_reliability(p, n):
    return p ** n
```

The takeaway: keep $n$ small (bounded loops, fewer steps), push $p$ up (validated tool I/O, retries on transient failures), and gate the irreversible steps with a human.
</details>

## Recap

- The model **emits** tool calls; your **runtime executes** them and returns observations.
- Reliability is $p^n$ — a 95% step is only ~60% over ten steps.
- Retries with bounded attempts recover transient tool failures ($1-f^k$).
- MCP standardises the *wiring* to tools; it does not make tool use *safe* — that stays your job.